# Choosing a game model
I began with home win rate, season record and net rating, then checked whether fuller basketball histories earned their place. This notebook excludes the final test season.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'Data').exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
TABLES = ROOT / 'artifacts/tables'
pd.set_option('display.max_columns', 12)

## How does it work in later seasons?
Six expanding windows validate 2017–18 through 2022–23. The first training window covers 2014–15 through 2016–17. Selection uses mean season log loss. Within 0.001 of the best, the fixed rule prefers fewer inputs, then logistic regression.

In [2]:
summary=pd.read_csv(TABLES / 'cv_summary.csv')
folds=pd.read_csv(TABLES / 'cv_folds.csv')
selection=json.loads((ROOT / 'artifacts/models/selection.json').read_text())
display(summary.head(10).round(4))
print('Selected:',selection['config'])
print('Training seasons:',selection['train_seasons'])

,name,log_loss,brier,auc,accuracy,season_sd,train_log_loss,kind,features,n_features
0,L1 combined20 C=0.1,0.6244,0.2173,0.6977,0.6519,0.0144,0.6034,logistic,combined20,24
1,L1 combined10 C=0.1,0.6247,0.2175,0.6968,0.6511,0.0149,0.6031,logistic,combined10,24
2,L2 combined20 C=0.01,0.6248,0.2175,0.6973,0.6528,0.0147,0.6033,logistic,combined20,24
3,L1 combined5 C=0.1,0.6249,0.2176,0.6963,0.6521,0.0149,0.6034,logistic,combined5,24
4,L1 team_players C=0.1,0.6250,0.2176,0.6967,0.6517,0.0148,0.6036,logistic,team_players,22
5,L2 combined10 C=0.01,0.6251,0.2177,0.6965,0.6512,0.0154,0.6031,logistic,combined10,24
6,L1 combined20 C=1,0.6253,0.2178,0.6965,0.6526,0.0147,0.6028,logistic,combined20,24
7,L2 combined20 C=0.1,0.6254,0.2178,0.6964,0.6525,0.0149,0.6028,logistic,combined20,24
8,L2 combined5 C=0.01,0.6255,0.2179,0.6958,0.6510,0.0153,0.6034,logistic,combined5,24
9,L1 combined10 C=1,0.6256,0.2179,0.6960,0.6498,0.0150,0.6027,logistic,combined10,24


Selected: {'name': 'L1 team_players C=0.1', 'kind': 'logistic', 'features': 'team_players', 'C': 0.1, 'l1_ratio': 1}
Training seasons: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


L1 with twenty-game team form and player histories has the lowest mean loss. Removing those two team-form inputs is only 0.00060 worse, so the prespecified rule selects the smaller version. That is a preference for parsimony, not evidence that recent form never matters.

## Which basketball histories helped?

In [3]:
best=pd.read_csv(TABLES / 'feature_comparison.csv').sort_values('log_loss')
display(best[['features','name','log_loss','brier','n_features']].round(4))
display(pd.read_csv(TABLES / 'player_comparison_by_season.csv').round(4))

,features,name,log_loss,brier,n_features
0,combined20,L1 combined20 C=0.1,0.6244,0.2173,24
1,combined10,L1 combined10 C=0.1,0.6247,0.2175,24
2,combined5,L1 combined5 C=0.1,0.6249,0.2176,24
3,team_players,L1 team_players C=0.1,0.6250,0.2176,22
4,recent20,L1 recent20 C=0.1,0.6267,0.2184,13
5,recent10,L1 recent10 C=0.1,0.6270,0.2185,13
6,compact,L2 compact C=0.01,0.6270,0.2185,6
7,recent5,L1 recent5 C=0.1,0.6276,0.2189,13
8,team,L1 team C=0.1,0.6281,0.2191,11
9,net_rating,Baseline net_rating,0.6306,0.2201,1


,season,Baseline net_rating,L1 team C=0.1,L1 team_players C=0.1,players_minus_team
0,2017,0.6145,0.6113,0.6065,-0.0048
1,2018,0.6215,0.6180,0.6143,-0.0036
2,2019,0.6246,0.6211,0.6209,-0.0002
3,2020,0.6327,0.6302,0.6244,-0.0058
4,2021,0.6419,0.6424,0.6371,-0.0053
5,2022,0.6483,0.6457,0.6466,0.0009


Player histories help the same L1 team model in five of six seasons, reducing mean loss from 0.6281 to 0.6250. Five-, ten- and twenty-game team form add smaller gains. These comparisons informed selection, so small differences can be optimistic.

## Does regularization keep it steadier?

In [4]:
reg=summary[(summary.features=='team_players') & (summary.kind=='logistic')]
display(reg[['name','log_loss','train_log_loss','season_sd']].round(4))
coef=pd.read_csv(TABLES / 'cv_coefficients.csv.gz')
stability=coef[coef.name.eq(selection['config']['name'])].groupby('feature').coefficient.agg(['mean','min','max'])
display(stability.loc[stability['mean'].abs().sort_values(ascending=False).head(10).index].round(3))
display(pd.read_csv(TABLES / 'feature_correlations.csv').head(8).round(3))

,name,log_loss,train_log_loss,season_sd
4,L1 team_players C=0.1,0.6250,0.6036,0.0148
12,L2 team_players C=0.01,0.6256,0.6037,0.0152
16,L1 team_players C=1,0.6258,0.6032,0.0150
17,L2 team_players C=0.1,0.6259,0.6032,0.0150
20,L2 team_players C=1,0.6260,0.6031,0.0149
21,Logistic team_players,0.6260,0.6031,0.0149
50,L1 team_players C=0.01,0.6279,0.6090,0.0110


,mean,min,max
feature,,,
net_rating_diff,0.451,0.424,0.504
win_pct_diff,0.281,0.229,0.319
back_to_back_diff,-0.091,-0.120,-0.059
scoring_concentration_diff,0.082,0.056,0.109
weighted_absence_diff,-0.072,-0.094,-0.057
weighted_ppa_diff,0.069,0.041,0.090
games_last7_diff,-0.064,-0.075,-0.055
efg_diff,0.059,0.017,0.089
dreb_rate_diff,0.034,0.021,0.048


,feature_a,feature_b,correlation
0,scoring_concentration_diff,usage_concentration_diff,0.947
1,net_rating_diff,win_pct_diff,0.939
2,top5_share_diff,top7_share_diff,0.926
3,weighted_absence_diff,weighted_participation_diff,-0.874
4,net_rating_diff,last10_net_rating_diff,0.794
5,efg_diff,last10_efg_diff,0.756
6,top5_share_diff,usage_concentration_diff,0.744
7,win_pct_diff,last10_net_rating_diff,0.742


Net rating and win percentage correlate at 0.939; scoring and usage concentration at 0.947. L1 can shrink coefficients to zero, but correlated inputs can exchange credit. Net rating, back-to-backs, scoring concentration, prior absences and weighted scoring keep consistent directions across development fits. Tiny or unstable coefficients should not become basketball stories.

## Did nonlinear models do better?

In [5]:
display(summary[summary.kind.isin(['forest','boosting'])][['name','train_log_loss','log_loss','brier']].round(4))
from src.modeling import make_model, metrics
dataset=pd.read_csv(TABLES / 'matchups.csv.gz')
train=dataset[dataset.eligible & (dataset.season<2022)]
valid=dataset[dataset.eligible & (dataset.season==2022)]
features=selection['features']
model=make_model(selection['config']).fit(train[features],train.y)
print('Reproduced last development window:',metrics(valid.y,model.predict_proba(valid[features])[:,1]))

,name,train_log_loss,log_loss,brier
59,Boosting combined10 leaves=3,0.5945,0.6287,0.2193
65,Boosting combined10 leaves=7,0.5700,0.6307,0.2199
66,Forest combined10 leaf=60,0.5628,0.6308,0.2199
67,Boosting team leaves=3,0.6004,0.6312,0.2204
68,Forest combined10 leaf=20,0.5246,0.6321,0.2204
69,Boosting team leaves=7,0.5824,0.6326,0.2208
70,Forest team leaf=60,0.5757,0.6339,0.2215
72,Forest team leaf=20,0.5437,0.6354,0.2219


Reproduced last development window: {'log_loss': 0.6466211944607494, 'brier': 0.22717766838550663, 'auc': 0.6537359081048402, 'accuracy': 0.6225535880708295, 'n': 1073}


The forest fits training games more closely but loses ground on later seasons. Shallow boosting is closer, yet trails regularized logistic regression. I keep the simpler model. Imputation and scaling are fitted within each training window. The selected model was committed before opening the 2023–24 evaluation.